In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import json
from dask.distributed import Client
import dask
import warnings
warnings.filterwarnings('ignore')
import platform, sys, os
import asf_search as asf
os.environ["PATH"] = "/home/pm4167/myGMTSAR/bin:" + os.environ["PATH"]



In [ ]:
# plotting modules
import pyvista as pv
# magic trick for white background
pv.set_plot_theme("document")
import panel
panel.extension(comms='ipywidgets')
panel.extension('vtk')
from contextlib import contextmanager
import matplotlib.pyplot as plt
@contextmanager
def mpl_settings(settings):
    original_settings = {k: plt.rcParams[k] for k in settings}
    plt.rcParams.update(settings)
    yield
    plt.rcParams.update(original_settings)
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.titlesize'] = 24
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
%matplotlib inline
# define Pandas display settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

In [1]:
#User Inputs
# Set these variables to None and you will be prompted to enter your username and password below.
asf_username = 'abcde***'
asf_password = 'defghi***'
ORBIT     = 'D'
SUBSWATH  = 1
REFERENCE = '2016-01-19' #20211111 20251226
BASELINE_Val=60
WORKDIR = 'China_data_asc_B60'  if ORBIT == 'D' else 'China_data_desc_B60'
DATADIR = 'data_China_ASC' if ORBIT == 'D' else 'data_China_DESC'
points = {'A1': [116.5695839774135, 40.10186014685715], 'A2': [116.5696349025209, 40.10136225503361], 'A3': [116.5697168017759, 40.10071592635534], 'A4': [116.5698804417336, 40.10023640249219], 'A5': [116.5699623347371, 40.09952753670751], 'A6': [116.5704968118338, 40.10012370387329], 'A7': [116.5717597584269, 40.10021187357275], 'A8': [116.5714408073707, 40.10174778393619], 'M1': [116.5948281669664, 40.089085316686], 'M2': [116.600682691875, 40.05338000222945], 'A9': [116.5691495564517, 40.10553429969959], 'A10': [116.5692768483404, 40.10489085062482], 'A11': [116.5693531863618, 40.10385886376077], 'A12': [116.5695313472745, 40.10308089451145], 'A13': [116.5703543629087, 40.09768114921123], 'A14': [116.5704688533819, 40.096776692365], 'A15': [116.5707425538325, 40.09529380747462], 'A16': [116.5711302718162, 40.09328754804627], 'A17': [116.5713355288918, 40.0917174209446], 'A18': [116.5714495558876, 40.09089745468683], 'A19': [116.5715863487555, 40.08974536435351], 'A20': [116.5718382034853, 40.08822088272657], 'A21': [116.572152127909, 40.10067090511745], 'A22': [116.5722113069274, 40.09982965377183], 'A23': [116.5723651000419, 40.09893375643752], 'A24': [116.573015280289, 40.10098721777899], 'A25': [116.5738079254336, 40.10102373108586], 'A26': [116.5739965341045, 40.10034499280381], 'A27': [116.5741510669756, 40.09933197106735], 'A28': [116.5731222557187, 40.09966675937559], 'A29': [116.5734534829844, 40.09818264205746], 'A30': [116.5742451565011, 40.09868980616866], 'A31': [116.5744455607867, 40.10112257115968], 'A32': [116.5745517522117, 40.10050759484754], 'A33': [116.5748472439109, 40.09980230576387], 'A34': [116.5748590445779, 40.09925073979848], 'A35': [116.5750481627487, 40.09888000990703], 'A36': [116.5749655135426, 40.10107723714383], 'A37': [116.5730152697199, 40.10035410244832], 'A38': [116.573931801161, 40.0957593294491], 'A39': [116.5743029666578, 40.09338108318717], 'A40': [116.5754129709537, 40.09098931632637], 'A41': [116.5768744631036, 40.09105072629666], 'A42': [116.5784172393485, 40.09126244444748], 'A43': [116.579881688216, 40.0904449390374], 'A44': [116.5781821063306, 40.09026288072035], 'A45': [116.5756898725413, 40.08992976086127], 'A46': [116.5773517819895, 40.08941501043698], 'B1': [116.6058195874324, 40.07630202962379], 'B2': [116.6060595963468, 40.07510685245862], 'B3': [116.6073817142677, 40.07699134707368], 'B4': [116.6040762389792, 40.07671524456642], 'B5': [116.6085894871276, 40.07787326721068], 'B6': [116.6024121625258, 40.07749606864195], 'B7': [116.6061936329149, 40.07422597273995], 'B8': [116.6060152813678, 40.07271871640322], 'B9': [116.6069341584596, 40.07261510316723], 'B10': [116.6062842367572, 40.07093713505358], 'B11': [116.6072018772635, 40.07103985193897], 'B12': [116.6095280039877, 40.05349060493358], 'B13': [116.6116661979754, 40.05349042752363], 'B14': [116.6069179675397, 40.05300959831899], 'B15': [116.6092134759395, 40.05558468191849], 'B16': [116.6089621957614, 40.05676395013406], 'B17': [116.6082784247793, 40.05827678671422], 'B18': [116.6091620571685, 40.05847639785728], 'B19': [116.6080053309372, 40.05996577542061], 'B20': [116.6088710309029, 40.06008900905367], 'B21': [116.607123979611, 40.06579991988892], 'B22': [116.6073126279835, 40.06446007981855], 'B23': [116.6080076994933, 40.06595654939285], 'B24': [116.6082093551017, 40.06452190330043], 'C1': [116.5911161751164, 40.09860825372954], 'C2': [116.5912645103345, 40.09791893915788], 'C3': [116.5913470503623, 40.09724455940966], 'C4': [116.5914489774072, 40.09663244285641], 'C5': [116.5915104424011, 40.09594213299377], 'C6': [116.5915924110301, 40.09520475576811], 'C7': [116.59167614923, 40.09453006693776], 'C8': [116.5918803103805, 40.09369846309062], 'D1': [116.5847021467645, 40.10211800812331], 'D2': [116.5848870222914, 40.10109974573128], 'D3': [116.5849694626307, 40.10052015279871], 'D4': [116.5849284265852, 40.09973617712594], 'D5': [116.5850511020701, 40.09926564170235], 'D6': [116.5851126340598, 40.09876396748334], 'D7': [116.5860937003641, 40.09631899943432], 'D8': [116.586197068888, 40.09573880249754], 'D9': [116.5862980440265, 40.09500309622253], 'D10': [116.5863605349871, 40.09448528793672], 'D11': [116.5845990558853, 40.09558213327501], 'D12': [116.5846605030681, 40.09517465676958], 'D13': [116.5847424356075, 40.09475150808475], 'D14': [116.5848250185603, 40.0942335275541], 'D15': [116.5795392526765, 40.0949558044309], 'D16': [116.5794983133104, 40.09537890412263], 'D17': [116.5783315530466, 40.09755685036338], 'D18': [116.5784122723647, 40.09658523552076], 'D19': [116.5789444790567, 40.09873220190054], 'D20': [116.5772029908889, 40.09862244224299], 'D21': [116.5783494844466, 40.10196140777776], 'D22': [116.5784522529972, 40.10153788149423], 'D23': [116.5760739364893, 40.09898332030802], 'D24': [116.5760739364893, 40.09898332030802], 'C9': [116.5951361351883, 40.09896858662891], 'C10': [116.5953843606997, 40.09781434881075], 'C11': [116.5954506460333, 40.09683288823743], 'C12': [116.5955762587002, 40.09582081957235], 'C13': [116.5959947684569, 40.09400568162772], 'D25': [116.5822046863219, 40.07971216229604], 'E1': [116.6062254724851, 40.10367951759758], 'E2': [116.6065410789498, 40.10218595880371], 'E3': [116.6068242255459, 40.10021327700974], 'E4': [116.6070507872574, 40.09882589822652], 'E5': [116.6072489453388, 40.09693992845735], 'E6': [116.6073054577595, 40.09550919261395], 'E7': [116.5784825890103, 40.08354134398889], 'E8': [116.5805021162266, 40.08378968044509], 'E9': [116.5831831775365, 40.08409872086018], 'E10': [116.5853796974262, 40.08430883803098], 'E11': [116.5869463328558, 40.08444477552844], 'E12': [116.5883837670915, 40.08456833959102], 'E13': [116.5896742532794, 40.08476501072984], 'E14': [116.5919681584508, 40.08485151936474], 'E15': [116.5921143882315, 40.08407377202987], 'E16': [116.5922114683011, 40.08332013710356], 'E17': [116.5931964872002, 40.08097225598696], 'E18': [116.59151682832, 40.08039153884177], 'E19': [116.5917428485488, 40.07871096741771], 'E20': [116.581778533032, 40.08184012758555], 'E21': [116.588454631597, 40.08318466207981], 'E22': [116.5849398550301, 40.08103913948992], 'E23': [116.5789521805304, 40.08110323597268], 'E24': [116.5772333092284, 40.08281803712828], 'D26': [116.582264010771, 40.07908356091982], 'D27': [116.5804888354486, 40.08107541811599], 'D28': [116.5834185863729, 40.08134177184059]}
# The subswath is required for partial scene downloads and is not used for burst downloads.
# The orbit is used to define directory names

In [ ]:
BURSTS = """
S1_099641_IW1_20160119T222039_VV_E137-BURST
S1_099641_IW1_20160212T222038_VV_5034-BURST
S1_099641_IW1_20160224T222038_VV_1459-BURST
S1_099641_IW1_20160307T222038_VV_724A-BURST
S1_099641_IW1_20160319T222038_VV_044E-BURST
S1_099641_IW1_20160412T222039_VV_4694-BURST
S1_099641_IW1_20160530T222042_VV_3F70-BURST
S1_099641_IW1_20160611T222042_VV_7E8A-BURST
S1_099641_IW1_20160705T222044_VV_1232-BURST
S1_099641_IW1_20160717T222044_VV_BE83-BURST
S1_099641_IW1_20160729T222045_VV_BDE3-BURST
S1_099641_IW1_20160810T222045_VV_F285-BURST
S1_099641_IW1_20160822T222046_VV_9229-BURST
S1_099641_IW1_20160903T222047_VV_864D-BURST
S1_099641_IW1_20160915T222047_VV_C54F-BURST
S1_099641_IW1_20161003T222005_VV_56FA-BURST
S1_099641_IW1_20161015T222005_VV_05DD-BURST
S1_099641_IW1_20161027T222006_VV_552F-BURST
S1_099641_IW1_20161108T222005_VV_B4E5-BURST
S1_099641_IW1_20161120T222005_VV_05E0-BURST
S1_099641_IW1_20161202T222005_VV_4D08-BURST
S1_099641_IW1_20161214T222004_VV_AB69-BURST
S1_099641_IW1_20161226T222004_VV_465B-BURST
S1_099641_IW1_20170107T222003_VV_9C65-BURST
S1_099641_IW1_20170119T222002_VV_CD9E-BURST
S1_099641_IW1_20170131T222002_VV_8390-BURST
S1_099641_IW1_20170212T222002_VV_7FBD-BURST
S1_099641_IW1_20170224T222002_VV_ADDA-BURST
S1_099641_IW1_20170308T222002_VV_9F77-BURST
S1_099641_IW1_20170320T222002_VV_655F-BURST
S1_099641_IW1_20170401T222002_VV_61A5-BURST
S1_099641_IW1_20170425T222003_VV_6463-BURST
S1_099641_IW1_20170507T222004_VV_F5F7-BURST
S1_099641_IW1_20170519T222005_VV_B492-BURST
S1_099641_IW1_20170531T222005_VV_73BE-BURST
S1_099641_IW1_20170612T222006_VV_0AE4-BURST
S1_099641_IW1_20170624T222007_VV_E04E-BURST
S1_099641_IW1_20170706T222007_VV_F042-BURST
S1_099641_IW1_20170718T222008_VV_B156-BURST
S1_099641_IW1_20170730T222009_VV_DC38-BURST
S1_099641_IW1_20170811T222009_VV_1688-BURST
S1_099641_IW1_20170823T222010_VV_39C3-BURST
S1_099641_IW1_20170916T222011_VV_E91C-BURST
S1_099641_IW1_20170928T222011_VV_8C9A-BURST
S1_099641_IW1_20171010T222011_VV_47B0-BURST
S1_099641_IW1_20171022T222012_VV_4F0A-BURST
S1_099641_IW1_20171103T222011_VV_6849-BURST
S1_099641_IW1_20171115T222011_VV_80F5-BURST
S1_099641_IW1_20171127T222011_VV_8558-BURST
S1_099641_IW1_20171209T222011_VV_319A-BURST
S1_099641_IW1_20171221T222010_VV_19E7-BURST
S1_099641_IW1_20180102T222009_VV_B3EE-BURST
S1_099641_IW1_20180207T222008_VV_E199-BURST
S1_099641_IW1_20180219T222008_VV_8E10-BURST
S1_099641_IW1_20180303T222008_VV_1934-BURST
S1_099641_IW1_20180315T222008_VV_C581-BURST
S1_099641_IW1_20180327T222009_VV_CEF0-BURST
S1_099641_IW1_20180408T222009_VV_0D3E-BURST
S1_099641_IW1_20180420T222009_VV_067B-BURST
S1_099641_IW1_20180502T222010_VV_359A-BURST
S1_099641_IW1_20180514T222010_VV_2541-BURST
S1_099641_IW1_20180526T222011_VV_8A3E-BURST
S1_099641_IW1_20180607T222012_VV_C1EB-BURST
S1_099641_IW1_20180619T222013_VV_9E91-BURST
S1_099641_IW1_20180701T222014_VV_AA07-BURST
S1_099641_IW1_20180713T222014_VV_8DC9-BURST
S1_099641_IW1_20180725T222015_VV_0FC3-BURST
S1_099641_IW1_20180806T222015_VV_FCC9-BURST
S1_099641_IW1_20180818T222016_VV_87B3-BURST
S1_099641_IW1_20180830T222017_VV_67B5-BURST
S1_099641_IW1_20180911T222017_VV_1E75-BURST
S1_099641_IW1_20180923T222018_VV_59EA-BURST
S1_099641_IW1_20181005T222018_VV_B31A-BURST
S1_099641_IW1_20181017T222018_VV_1829-BURST
S1_099641_IW1_20181029T222018_VV_2892-BURST
S1_099641_IW1_20181110T222018_VV_DDDE-BURST
S1_099641_IW1_20181122T222018_VV_3665-BURST
S1_099641_IW1_20181204T222017_VV_7E13-BURST
S1_099641_IW1_20181216T222017_VV_00F2-BURST
S1_099641_IW1_20181228T222017_VV_D4FE-BURST
S1_099641_IW1_20190109T222016_VV_C391-BURST
S1_099641_IW1_20190121T222016_VV_D568-BURST
S1_099641_IW1_20190202T222015_VV_22E5-BURST
S1_099641_IW1_20190214T222015_VV_5E79-BURST
S1_099641_IW1_20190226T222015_VV_E4FC-BURST
S1_099641_IW1_20190310T222015_VV_DAC6-BURST
S1_099641_IW1_20190322T222015_VV_0554-BURST
S1_099641_IW1_20190403T222015_VV_DD15-BURST
S1_099641_IW1_20190415T222016_VV_F8B6-BURST
S1_099641_IW1_20190427T222016_VV_8B9E-BURST
S1_099641_IW1_20190509T222017_VV_5B51-BURST
S1_099641_IW1_20190521T222017_VV_26B0-BURST
S1_099641_IW1_20190602T222018_VV_1AE0-BURST
S1_099641_IW1_20190813T222022_VV_52BD-BURST
S1_099641_IW1_20190825T222023_VV_8529-BURST
S1_099641_IW1_20190918T222024_VV_1A09-BURST
S1_099641_IW1_20190930T222025_VV_0E16-BURST
S1_099641_IW1_20191024T222025_VV_CED1-BURST
S1_099641_IW1_20191105T222024_VV_2AE6-BURST
S1_099641_IW1_20191117T222024_VV_B76A-BURST
S1_099641_IW1_20191129T222024_VV_839E-BURST
S1_099641_IW1_20191211T222024_VV_C564-BURST
"""
BURSTS = list(filter(None, BURSTS.split('\n')))
print (f'Bursts defined: {len(BURSTS)}')

# select the only 1A satellite bursts corresponding to the scenes above
# import re
# scene_dates = [re.search(r'\d{8}T\d{6}', scene).group(0)[:8] for scene in SCENES]
# burst_dates = [re.search(r'\d{8}T\d{6}', burst).group(0)[:8] for burst in BURSTS]
# matching_bursts = [burst for burst in BURSTS if any(date in burst for date in scene_dates)]
# for burst in matching_bursts: print (burst)

In [ ]:
# Define search parameters
results = asf.search(
    dataset=asf.DATASET.SLC_BURST,
    start='2016-01-01',
    end='2019-12-31',
    intersectsWith='POLYGON((116.54 40.11,116.54 40.04,116.66 40.04,116.66 40.11,116.54 40.11))',
    flightDirection='DESCENDING', #DESCENDING
    polarization='VV'#,    fileID='*S1_277224_IW1*'
)

# Sort results by date (ascending)
sorted_results = sorted(results, key=lambda r: r.properties['startTime'])

# Print download IDs in ascending date order
for result in sorted_results:
    fid = result.properties['fileID']
    #print(fid)
    if 'S1_099641_IW1' in fid: #130_277225_IW1
        print(fid)

#for result in sorted_results:
#  if result.properties['fileID'] == '*S1_277224_IW1*':
 #   print(result.properties['fileID'])


In [ ]:
#Define AOI and points
geojson = '''
{
  "type": "Feature",
  "properties": {},
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [116.54, 40.11],[116.66, 40.11],[116.66, 40.04],[116.54, 40.04], [116.54, 40.11]
      ]
    ]
  }
}
'''
AOI = gpd.GeoDataFrame.from_features([json.loads(geojson)])

# subsidence point from https://blog.descarteslabs.com/sentinel-1-targeted-analysis
geojson = '''
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates":  [116.6006827, 40.0533800]
  },
  "properties": {}
}
'''
POI = gpd.GeoDataFrame.from_features([json.loads(geojson)])
geojson = '''
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [116.5948282, 40.0890853]
  },
  "properties": {}
}
'''
POI0 = gpd.GeoDataFrame.from_features([json.loads(geojson)])
POI0

In [ ]:

from pygmtsar import S1, Stack, tqdm_dask, ASF, Tiles, XYZTiles

# recent Google Colab changes in early September 2025 broke Dask+Xarray NetCDF multithedded processing (again)
# workaround below disables multitheading when it does not work, degrading performance and increasing RAM usage.
if 'google.colab' in sys.modules:
    methods = {
        "load_dem":  "synchronous",
        "save_cube": "compute",
        "save_stack":"compute",
    }
    for m, kind in methods.items():
        if not hasattr(Stack, f"_{m}"):
            setattr(Stack, f"_{m}", getattr(Stack, m))
        def _make_wrapper(name, kind):
            orig = getattr(Stack, f"_{name}")
            if kind == "synchronous":
                def _wrapper(self, *args, **kwargs):
                    with dask.config.set(scheduler="synchronous"):
                        return orig(self, *args, **kwargs)
                return _wrapper
            elif kind == "compute":
                def _wrapper(self, *args, **kwargs):
                    if args:
                        return orig(self, args[0].compute() if hasattr(args[0], "compute") else args[0], *args[1:], **kwargs)
                    return orig(self, **kwargs)
                return _wrapper
            else:
                raise NotImplementedError(f"Unknown wrapper kind: {kind}")
        setattr(Stack, m, _make_wrapper(m, kind))

In [ ]:
# define DEM and landmask filenames inside data directory
DEM = f'{DATADIR}/dem.nc'

In [ ]:
BUFFER = 0.025
# geometry is too small for the processing, enlarge it
AOI['geometry'] = AOI.buffer(BUFFER)

In [ ]:
# Set these variables to None and you will be prompted to enter your username and password below.
asf = ASF(asf_username, asf_password)
# Optimized scene downloading from ASF - only the required subswaths and polarizations.
# Subswaths are already encoded in burst identifiers and are only needed for scenes.
#print(asf.download(DATADIR, SCENES, SUBSWATH))
print(asf.download(DATADIR, BURSTS))

In [ ]:
# scan the data directory for SLC scenes and download missed orbits
S1.download_orbits(DATADIR, S1.scan_slc(DATADIR))

In [ ]:
# download NASA SRTM DEM 1 arc-second
Tiles().download_dem_srtm(AOI, filename=DEM).plot.imshow(cmap='cividis')

## Run Local Dask Cluster

Launch Dask cluster for local and distributed multicore computing. That's possible to process terabyte scale Sentinel-1 SLC datasets on Apple Air 16 GB RAM.

In [ ]:
# simple Dask initialization
if 'client' in globals():
    client.close()
client = Client()
client

## Init

Search recursively for measurement (.tiff) and annotation (.xml) and orbit (.EOF) files in the DATA directory. It can be directory with full unzipped scenes (.SAFE) subdirectories or just a directory with the list of pairs of required .tiff and .xml files (maybe pre-filtered for orbit, polarization and subswath to save disk space). If orbit files and DEM are missed these will be downloaded automatically below.

### Select Original Secenes and Orbits

Use filters to find required subswath, polarization and orbit in original scenes .SAFE directories in the data directory.

In [ ]:
scenes = S1.scan_slc(DATADIR, subswath=SUBSWATH)

In [ ]:
sbas = Stack(WORKDIR, drop_if_exists=True).set_scenes(scenes).set_reference(REFERENCE)
sbas.to_dataframe()

In [ ]:
sbas.plot_scenes(AOI=AOI)

## Reframe Scenes (Optional)

Stitch sequential scenes and crop the subswath to a smaller area for faster processing when the full area is not needed.

In [ ]:
sbas.compute_reframe(AOI)

In [ ]:
sbas.plot_scenes(AOI=AOI)

### Load DEM

The function below loads DEM from file or Xarray variable and converts heights to ellipsoidal model using EGM96 grid.

In [ ]:
# define the area of interest (AOI) to speedup the processing
sbas.load_dem(DEM, AOI)

In [ ]:
sbas.plot_scenes(AOI=AOI)

## Align Images

In [ ]:
sbas.compute_align()

## Geocoding Transform

In [ ]:
# use the original Sentinel-1 resolution (1 pixel spacing)
sbas.compute_geocode(1)

In [ ]:
sbas.plot_topo(quantile=[0.01, 0.99])

## SBAS Baseline

In [ ]:
baseline_pairs = sbas.sbas_pairs(days=BASELINE_Val)
# optionally, drop dates having less then 260 pairs
#baseline_pairs = sbas.sbas_pairs_limit(baseline_pairs, limit=2, iterations=2)
# optionally, drop all pairs connected to the specified dates
#baseline_pairs = sbas.sbas_pairs_filter_dates(baseline_pairs, ['2021-01-01'])
baseline_pairs

In [ ]:
with mpl_settings({'figure.dpi': 300}):
    sbas.plot_baseline(baseline_pairs)

In [ ]:
with mpl_settings({
    'figure.dpi': 300,
    'font.size': 0
}):
    sbas.plot_baseline(baseline_pairs)

## Persistent Scatterers Function (PSF)

In [ ]:
# use the only selected dates for the pixels stability analysis
sbas.compute_ps()

In [ ]:
sbas.plot_psfunction(quantile=[0.01, 0.90])

## SBAS Analysis

### Multi-looked Resolution for SBAS

In [ ]:
sbas.compute_interferogram_multilook(baseline_pairs, 'intf_mlook', wavelength=30, weight=sbas.psfunction())

In [ ]:
ds_sbas = sbas.open_stack('intf_mlook')
intf_sbas = ds_sbas.phase
corr_sbas = ds_sbas.correlation
corr_sbas

In [ ]:
intf_sbas

In [ ]:
sbas.plot_interferograms(intf_sbas[:8], caption='SBAS Phase, [rad]')

In [ ]:
sbas.plot_correlations(corr_sbas[:8], caption='SBAS Correlation')

### 2D Unwrapping

In [ ]:
unwrap_sbas = sbas.unwrap_snaphu(intf_sbas, corr_sbas)
unwrap_sbas

In [ ]:
# optionally, materialize to disk and open
unwrap_sbas = sbas.sync_cube(unwrap_sbas, 'unwrap_sbas')

In [ ]:
sbas.plot_phases(unwrap_sbas.phase[:8], caption='SBAS Phase, [rad]')

### Trend Correction

In [ ]:
decimator = sbas.decimator(resolution=15, grid=(1,1))
topo = decimator(sbas.get_topo())
inc = decimator(sbas.incidence_angle())
yy, xx = xr.broadcast(topo.y, topo.x)
trend_sbas = sbas.regression(unwrap_sbas.phase,
        [topo,    topo*yy,    topo*xx,    topo*yy*xx,
         topo**2, topo**2*yy, topo**2*xx, topo**2*yy*xx,
         topo**3, topo**3*yy, topo**3*xx, topo**3*yy*xx,
         inc,     inc**yy,    inc*xx,     inc*yy*xx,
         yy, xx,
         yy**2, xx**2, yy*xx,
         yy**3, xx**3, yy**2*xx, xx**2*yy], corr_sbas)

In [ ]:
# optionally, materialize to disk and open
trend_sbas = sbas.sync_cube(trend_sbas, 'trend_sbas')

In [ ]:
sbas.plot_phases(trend_sbas[:8], caption='SBAS Trend Phase, [rad]', quantile=[0.01, 0.99])

In [ ]:
sbas.plot_phases((unwrap_sbas.phase - trend_sbas)[:8], caption='SBAS Phase - Trend, [rad]', vmin=-np.pi, vmax=np.pi)

### Coherence-Weighted Least-Squares Solution for LOS Displacement, mm

In [ ]:
# calculate phase displacement in radians and convert to LOS displacement in millimeter
disp_sbas = sbas.los_displacement_mm(sbas.lstsq(unwrap_sbas.phase - trend_sbas, corr_sbas))

In [ ]:
# optionally, materialize to disk and open
disp_sbas = sbas.sync_cube(disp_sbas, 'disp_sbas')

In [ ]:
sbas.plot_displacements(disp_sbas[::3], caption='SBAS Cumulative LOS Displacement, [mm]', quantile=[0.01, 0.99])

### Least-squares model for LOS Displacement, mm

In [ ]:
velocity_sbas = sbas.velocity(disp_sbas)
velocity_sbas

In [ ]:
# optionally, materialize to disk and open
velocity_sbas = sbas.sync_cube(velocity_sbas, 'velocity_sbas')

In [ ]:
fig = plt.figure(figsize=(12,4), dpi=300)

zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

ax = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
sbas.geocode(AOI.buffer(-BUFFER).boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
sbas.geocode(POI0).plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

ax = fig.add_subplot(1, 2, 2)
sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry.buffer(-BUFFER))\
    .plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
AOI.buffer(-BUFFER).boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
POI0.plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

plt.suptitle('SBAS LOS Velocity, 2021', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12,4), dpi=300)

zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

ax = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(
    cmap='turbo',
    vmin=-zminmax,
    vmax=zminmax,
    ax=ax
)

sbas.geocode(AOI.buffer(-BUFFER).boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
sbas.geocode(POI0).plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')

ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

ax = fig.add_subplot(1, 2, 2)

sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(
    AOI.geometry.buffer(-BUFFER)
).plot.imshow(
    cmap='turbo',
    vmin=-zminmax,
    vmax=zminmax,
    ax=ax
)

AOI.buffer(-BUFFER).boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
POI0.plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')

ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

plt.suptitle('SBAS Descending LOS Velocity', fontsize=18)

plt.tight_layout()

# Save high-resolution figure
plt.savefig(
    'SBAS_LOS_Velocity_B60_Desc.png',
    dpi=600,
    bbox_inches='tight',
    pad_inches=0.1
)

plt.show()

### STL model for LOS Displacement, mm

In [ ]:
plt.figure(figsize=(12, 4), dpi=300)

x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI).geometry][0]
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)
plt.plot(disp_pixel.date, disp_pixel, c='r', lw=2, label='Displacement POI')
plt.plot(stl_pixel.date, stl_pixel.trend, c='r', ls='--', lw=2, label='Trend POI')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='r', lw=1, label='Seasonal POI')

x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI0).geometry][0]
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)
plt.plot(disp_pixel.date, disp_pixel, c='b', lw=2, label='Displacement POI$\Theta$')
plt.plot(stl_pixel.date, stl_pixel.trend, c='b', ls='--', lw=2, label='Trend POI$\Theta$')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='b', lw=1, label='Seasonal POI$\Theta$')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=14)
plt.title('SBAS LOS Displacement STL Decompose, 2021', fontsize=18)
plt.ylabel('Displacement, mm', fontsize=16)
plt.show()

In [ ]:
# ---------------------------------
# Select only specific POIs
# ---------------------------------
selected_pois = ["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12","A13","A14","A15","A16","A17","A18","A19","A20","A21","A22","A23","A24","A25","A26","A27","A28","A29","A30","A31","A32","A33","A34","A35","A36","A37","A38","A39","A40","A41","A42","A43","A44","A45","A46"]   # <-- put desired names here

# Filter dictionary
filtered_points = {k: v for k, v in points.items() if k in selected_pois}

colors = plt.cm.tab20(np.linspace(0, 1, len(filtered_points)))

plt.figure(figsize=(14, 6), dpi=300)

for (name, coord), color in zip(filtered_points.items(), colors):

    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }

    gdf = gpd.GeoDataFrame.from_features([geojson])
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    disp_px = disp_sbas.sel(x=x, y=y, method="nearest")

    plt.plot(disp_px.date, disp_px,
             lw=1.8,
             color=color,
             label=f"{name} Displacement")

plt.title("SBAS Displacement Time Series for Selected POIs")
plt.xlabel("Timeline")
plt.ylabel("Displacement [mm]")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(ncol=2, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
import dask

with dask.config.set(scheduler="synchronous"):
    disp_sbas.to_netcdf(
        "disp_sbas_CHNA_2014_2019_Desc_B60_May20.nc"
    )



In [ ]:
import geopandas as gpd
import pandas as pd
import json
import numpy as np

# -------------------------------
# Geometry grids from PyGMTSAR
# -------------------------------

# incidence angle (radians)
inc = sbas.incidence_angle()

# satellite look vector
lv = sbas.get_satellite_look_vector()
look_E = lv["look_E"]
look_N = lv["look_N"]
look_U = lv["look_U"]

records = []

for name, coord in points.items():

    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }
    gdf = gpd.GeoDataFrame.from_features([geojson])

    # lon/lat → radar x,y
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    # sample incidence angle
    inc_rad = inc.sel(x=x, y=y, method="nearest").values.item()
    inc_deg = np.rad2deg(inc_rad)

    # sample look vector components
    le = look_E.sel(x=x, y=y, method="nearest").values.item()
    ln = look_N.sel(x=x, y=y, method="nearest").values.item()
    lu = look_U.sel(x=x, y=y, method="nearest").values.item()

    records.append({
        "point": name,
        "lon": coord[0],
        "lat": coord[1],
        "x": x,
        "y": y,

        # incidence
        "incidence_rad": inc_rad,
        "incidence_deg": inc_deg,

        # look vector components (ENU)
        "look_E": le,
        "look_N": ln,
        "look_U": lu
    })

df_geom_desc = pd.DataFrame(records)

# save for later restore-only use
df_geom_desc.to_csv(
    "Apoi_geometryALLPoints_angles_CHNA_2016_2019_DESC_B60_May20.csv",
    index=False
)




In [ ]:
# ============================================================
# Optimized vs Non-Optimized SBAS Network Comparison
# Fair Computational Timing + Sensitivity Analysis
# Insert after:
#     sbas.compute_ps()
# ============================================================

import os
import gc
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import dask
import xarray as xr

# ------------------------------------------------------------
# 0. User-defined SBAS constraints
# ------------------------------------------------------------

MAX_DAYS_NONOPT = 60
MAX_BPERP_NONOPT = 150

MAX_DAYS_OPT = 60
MAX_BPERP_OPT = 150

LIMIT_VALUES = [1, 2, 3, 4, 5]
N_ITERATIONS = 3
OPTIMAL_LIMIT = 2

N_RUNTIME_REPEATS = 2

OUTPUT_DIR = "sbas_network_comparison_outputs_DESC"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 1. Utility functions
# ------------------------------------------------------------

def pairs_to_graph(pairs):
    G = nx.Graph()

    if hasattr(pairs, "columns"):
        cols = list(pairs.columns)

        if "ref" in cols and "rep" in cols:
            ref_col, rep_col = "ref", "rep"
        elif "reference" in cols and "repeat" in cols:
            ref_col, rep_col = "reference", "repeat"
        elif "date1" in cols and "date2" in cols:
            ref_col, rep_col = "date1", "date2"
        else:
            ref_col, rep_col = cols[0], cols[1]

        for _, row in pairs.iterrows():
            G.add_edge(str(row[ref_col]), str(row[rep_col]))
    else:
        for p in pairs:
            G.add_edge(str(p[0]), str(p[1]))

    return G


def network_statistics(G, pairs, name, limit_value=None, runtime=None):
    degrees = np.array([d for _, d in G.degree()])

    return {
        "network": name,
        "limit": limit_value,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "connected": nx.is_connected(G) if G.number_of_nodes() > 0 else False,
        "components": nx.number_connected_components(G) if G.number_of_nodes() > 0 else np.nan,
        "average_degree": np.mean(degrees) if len(degrees) > 0 else np.nan,
        "minimum_degree": np.min(degrees) if len(degrees) > 0 else np.nan,
        "maximum_degree": np.max(degrees) if len(degrees) > 0 else np.nan,
        "max_abs_baseline_m": np.abs(pairs["baseline"]).max() if "baseline" in pairs.columns else np.nan,
        "mean_abs_baseline_m": np.abs(pairs["baseline"]).mean() if "baseline" in pairs.columns else np.nan,
        "max_duration_days": pairs["duration"].max() if "duration" in pairs.columns else np.nan,
        "mean_duration_days": pairs["duration"].mean() if "duration" in pairs.columns else np.nan,
        "network_generation_runtime_s": runtime
    }


def remove_existing_stack(stack_name):
    possible_paths = [
        stack_name,
        f"{stack_name}.grd",
        f"{stack_name}.nc",
        os.path.join(OUTPUT_DIR, stack_name),
    ]

    for path in possible_paths:
        if os.path.isdir(path):
            shutil.rmtree(path, ignore_errors=True)
        elif os.path.isfile(path):
            os.remove(path)


def clean_memory():
    gc.collect()


# ------------------------------------------------------------
# 2. Generate non-optimized and optimized candidate networks
# ------------------------------------------------------------

t0 = time.perf_counter()
baseline_pairs_nonopt = sbas.sbas_pairs(
    days=MAX_DAYS_NONOPT,
    meters=MAX_BPERP_NONOPT
)
runtime_nonopt_pair_selection = time.perf_counter() - t0

t0 = time.perf_counter()
baseline_pairs_opt_raw = sbas.sbas_pairs(
    days=MAX_DAYS_OPT,
    meters=MAX_BPERP_OPT
)
runtime_opt_raw_pair_selection = time.perf_counter() - t0

print("Non-optimized candidate pairs:", len(baseline_pairs_nonopt))
print("Optimized raw candidate pairs:", len(baseline_pairs_opt_raw))


# ------------------------------------------------------------
# 3. Sensitivity analysis for redundancy-control limit
# ------------------------------------------------------------

sensitivity_rows = []
optimized_pairs_by_limit = {}
optimization_runtime_by_limit = {}

for limit_value in LIMIT_VALUES:
    print(f"\nRunning network sensitivity for limit = {limit_value}")

    t0 = time.perf_counter()

    pairs_limited = sbas.sbas_pairs_limit(
        baseline_pairs_opt_raw,
        limit=limit_value,
        iterations=N_ITERATIONS
    )

    runtime_limit = time.perf_counter() - t0

    G_limited = pairs_to_graph(pairs_limited)

    stats = network_statistics(
        G_limited,
        pairs_limited,
        name=f"Optimized_limit_{limit_value}",
        limit_value=limit_value,
        runtime=runtime_limit
    )

    sensitivity_rows.append(stats)
    optimized_pairs_by_limit[limit_value] = pairs_limited
    optimization_runtime_by_limit[limit_value] = runtime_limit

df_sensitivity = pd.DataFrame(sensitivity_rows)

df_sensitivity_out = df_sensitivity[
    [
        "limit",
        "nodes",
        "edges",
        "components",
        "connected",
        "average_degree",
        "minimum_degree",
        "maximum_degree",
        "max_abs_baseline_m",
        "mean_abs_baseline_m",
        "max_duration_days",
        "mean_duration_days",
        "network_generation_runtime_s"
    ]
]

df_sensitivity_out.to_csv(
    os.path.join(OUTPUT_DIR, "network_optimization_sensitivity_limits.csv"),
    index=False
)

print("\nNetwork optimization sensitivity table")
print(df_sensitivity_out)


# ------------------------------------------------------------
# 4. Select optimized network
# ------------------------------------------------------------

baseline_pairs_opt = optimized_pairs_by_limit[OPTIMAL_LIMIT]
runtime_opt_limit_selection = optimization_runtime_by_limit[OPTIMAL_LIMIT]

print("\nSelected optimized network")
print("Optimal limit:", OPTIMAL_LIMIT)
print("Optimized pairs:", len(baseline_pairs_opt))


# ------------------------------------------------------------
# 5. Network diagnostics
# ------------------------------------------------------------

G_nonopt = pairs_to_graph(baseline_pairs_nonopt)
G_opt = pairs_to_graph(baseline_pairs_opt)

stats_nonopt = network_statistics(
    G_nonopt,
    baseline_pairs_nonopt,
    "Non-optimized",
    limit_value=np.nan,
    runtime=runtime_nonopt_pair_selection
)

stats_opt = network_statistics(
    G_opt,
    baseline_pairs_opt,
    f"Optimized_limit_{OPTIMAL_LIMIT}",
    limit_value=OPTIMAL_LIMIT,
    runtime=runtime_opt_raw_pair_selection + runtime_opt_limit_selection
)

df_network_comparison = pd.DataFrame([stats_nonopt, stats_opt])

df_network_comparison_out = df_network_comparison[
    [
        "network",
        "limit",
        "nodes",
        "edges",
        "components",
        "connected",
        "average_degree",
        "minimum_degree",
        "maximum_degree",
        "max_abs_baseline_m",
        "mean_abs_baseline_m",
        "max_duration_days",
        "mean_duration_days",
        "network_generation_runtime_s"
    ]
]

df_network_comparison_out.to_csv(
    os.path.join(OUTPUT_DIR, "network_comparison_nonoptimized_vs_optimized.csv"),
    index=False
)

print("\nNetwork comparison table")
print(df_network_comparison_out)


# ------------------------------------------------------------
# 6. Plot sensitivity metrics
# ------------------------------------------------------------

plt.figure(figsize=(8, 4), dpi=300)
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["components"], marker="o")
plt.xlabel("Redundancy-control limit")
plt.ylabel("Number of connected components")
plt.title("Connected Components vs Redundancy-Control Limit")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sensitivity_components_vs_limit.png"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4), dpi=300)
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["average_degree"], marker="o", label="Average degree")
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["minimum_degree"], marker="s", label="Minimum degree")
plt.xlabel("Redundancy-control limit")
plt.ylabel("Node degree")
plt.title("Network Redundancy vs Redundancy-Control Limit")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sensitivity_degree_vs_limit.png"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4), dpi=300)
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["network_generation_runtime_s"], marker="o")
plt.xlabel("Redundancy-control limit")
plt.ylabel("Optimization runtime [s]")
plt.title("Network Optimization Runtime")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sensitivity_runtime_vs_limit.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 7. Plot baseline networks
# ------------------------------------------------------------

with mpl_settings({"figure.dpi": 300}):
    sbas.plot_baseline(baseline_pairs_nonopt)
    plt.title(
        f"Non-Optimized SBAS Network\n"
        f"Temporal <= {MAX_DAYS_NONOPT} days, |Bperp| <= {MAX_BPERP_NONOPT} m"
    )
    plt.savefig(os.path.join(OUTPUT_DIR, "baseline_network_nonoptimized.png"), dpi=300, bbox_inches="tight")
    plt.show()

with mpl_settings({"figure.dpi": 300}):
    sbas.plot_baseline(baseline_pairs_opt)
    plt.title(
        f"Optimized SBAS Network\n"
        f"Limit = {OPTIMAL_LIMIT}, Iterations = {N_ITERATIONS}, "
        f"Temporal <= {MAX_DAYS_OPT} days, |Bperp| <= {MAX_BPERP_OPT} m"
    )
    plt.savefig(os.path.join(OUTPUT_DIR, "baseline_network_optimized.png"), dpi=300, bbox_inches="tight")
    plt.show()


# ------------------------------------------------------------
# 8. Full SBAS processing function with timing
# ------------------------------------------------------------

def run_sbas_network(pairs, label, run_id):
    print(f"\nRunning SBAS network: {label}, run {run_id}")

    timings = {}

    intf_name = f"intf_mlook_{label}_run{run_id}"
    unwrap_name = f"unwrap_{label}_run{run_id}"
    trend_name = f"trend_{label}_run{run_id}"
    disp_name = f"disp_{label}_run{run_id}"
    velocity_name = f"velocity_{label}_run{run_id}"

    for name in [intf_name, unwrap_name, trend_name, disp_name, velocity_name]:
        remove_existing_stack(name)

    clean_memory()

    total_start = time.perf_counter()

    # Interferogram generation
    t0 = time.perf_counter()

    sbas.compute_interferogram_multilook(
        pairs,
        intf_name,
        wavelength=30,
        weight=sbas.psfunction()
    )

    ds = sbas.open_stack(intf_name)
    intf = ds.phase
    corr = ds.correlation

    timings["interferogram_generation_s"] = time.perf_counter() - t0

    # Phase unwrapping
    t0 = time.perf_counter()

    unwrap = sbas.unwrap_snaphu(intf, corr)
    unwrap = sbas.sync_cube(unwrap, unwrap_name)

    timings["phase_unwrapping_s"] = time.perf_counter() - t0

    # Trend correction
    t0 = time.perf_counter()

    decimator = sbas.decimator(resolution=15, grid=(1, 1))
    topo = decimator(sbas.get_topo())
    inc = decimator(sbas.incidence_angle())
    yy, xx = xr.broadcast(topo.y, topo.x)

    trend = sbas.regression(
        unwrap.phase,
        [
            topo, topo * yy, topo * xx, topo * yy * xx,
            topo**2, topo**2 * yy, topo**2 * xx, topo**2 * yy * xx,
            topo**3, topo**3 * yy, topo**3 * xx, topo**3 * yy * xx,
            inc, inc * yy, inc * xx, inc * yy * xx,
            yy, xx,
            yy**2, xx**2, yy * xx,
            yy**3, xx**3, yy**2 * xx, xx**2 * yy
        ],
        corr
    )

    trend = sbas.sync_cube(trend, trend_name)

    timings["trend_correction_s"] = time.perf_counter() - t0

    # Least-squares inversion
    t0 = time.perf_counter()

    disp = sbas.los_displacement_mm(
        sbas.lstsq(unwrap.phase - trend, corr)
    )

    disp = sbas.sync_cube(disp, disp_name)

    timings["time_series_inversion_s"] = time.perf_counter() - t0

    # Velocity estimation
    t0 = time.perf_counter()

    velocity = sbas.velocity(disp)
    velocity = sbas.sync_cube(velocity, velocity_name)

    timings["velocity_estimation_s"] = time.perf_counter() - t0

    timings["total_processing_s"] = time.perf_counter() - total_start

    return {
        "label": label,
        "run_id": run_id,
        "pairs": pairs,
        "intf": intf,
        "corr": corr,
        "unwrap": unwrap,
        "trend": trend,
        "disp": disp,
        "velocity": velocity,
        "timings": timings
    }


# ------------------------------------------------------------
# ------------------------------------------------------------
# 9. Fair runtime experiment
# Optimized network is always run before non-optimized network
# ------------------------------------------------------------

runtime_results = []
final_result_nonopt = None
final_result_opt = None

for run_id in range(1, N_RUNTIME_REPEATS + 1):

    run_order = [
        ("opt", baseline_pairs_opt, f"Optimized_limit_{OPTIMAL_LIMIT}"),
        ("nonopt", baseline_pairs_nonopt, "Non-optimized")
    ]

    for order_position, (short_label, pairs, network_name) in enumerate(run_order, start=1):

        result = run_sbas_network(pairs, short_label, run_id)

        row = {
            "network": network_name,
            "run_id": run_id,
            "run_order_position": order_position,
            "pairs": len(pairs)
        }

        row.update(result["timings"])
        runtime_results.append(row)

        if short_label == "nonopt":
            final_result_nonopt = result
        else:
            final_result_opt = result

df_runtime_all = pd.DataFrame(runtime_results)

df_runtime_all.to_csv(
    os.path.join(OUTPUT_DIR, "runtime_all_repeats_nonoptimized_vs_optimized.csv"),
    index=False
)

print("\nAll runtime repeats")
print(df_runtime_all)


# ------------------------------------------------------------
# 10. Runtime summary
# ------------------------------------------------------------

timing_columns = [
    "interferogram_generation_s",
    "phase_unwrapping_s",
    "trend_correction_s",
    "time_series_inversion_s",
    "velocity_estimation_s",
    "total_processing_s"
]

df_runtime_summary = df_runtime_all.groupby("network").agg(
    pairs=("pairs", "mean"),
    **{f"{col}_mean": (col, "mean") for col in timing_columns},
    **{f"{col}_std": (col, "std") for col in timing_columns}
).reset_index()

nonopt_total_mean = df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "total_processing_s_mean"
].iloc[0]

nonopt_pairs_mean = df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "pairs"
].iloc[0]

df_runtime_summary["runtime_reduction_relative_to_nonopt_percent"] = (
    (nonopt_total_mean - df_runtime_summary["total_processing_s_mean"])
    / nonopt_total_mean * 100.0
)

df_runtime_summary["pair_reduction_relative_to_nonopt_percent"] = (
    (nonopt_pairs_mean - df_runtime_summary["pairs"])
    / nonopt_pairs_mean * 100.0
)

# Add pair-selection and optimization overhead
df_runtime_summary["network_generation_overhead_s"] = np.nan

df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "network_generation_overhead_s"
] = runtime_nonopt_pair_selection

df_runtime_summary.loc[
    df_runtime_summary["network"] == f"Optimized_limit_{OPTIMAL_LIMIT}",
    "network_generation_overhead_s"
] = runtime_opt_raw_pair_selection + runtime_opt_limit_selection

df_runtime_summary["total_processing_with_network_generation_s"] = (
    df_runtime_summary["total_processing_s_mean"]
    + df_runtime_summary["network_generation_overhead_s"]
)

nonopt_total_with_overhead = df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "total_processing_with_network_generation_s"
].iloc[0]

df_runtime_summary["runtime_reduction_with_overhead_relative_to_nonopt_percent"] = (
    (nonopt_total_with_overhead - df_runtime_summary["total_processing_with_network_generation_s"])
    / nonopt_total_with_overhead * 100.0
)

df_runtime_summary.to_csv(
    os.path.join(OUTPUT_DIR, "computational_efficiency_summary_nonoptimized_vs_optimized.csv"),
    index=False
)

print("\nRuntime summary")
print(df_runtime_summary)


# ------------------------------------------------------------
# 11. Interpretation flag
# ------------------------------------------------------------

opt_name = f"Optimized_limit_{OPTIMAL_LIMIT}"

opt_runtime_reduction = df_runtime_summary.loc[
    df_runtime_summary["network"] == opt_name,
    "runtime_reduction_with_overhead_relative_to_nonopt_percent"
].iloc[0]

opt_pair_reduction = df_runtime_summary.loc[
    df_runtime_summary["network"] == opt_name,
    "pair_reduction_relative_to_nonopt_percent"
].iloc[0]

if opt_runtime_reduction > 0:
    efficiency_statement = (
        "The optimized network reduced total runtime relative to the non-optimized network "
        "after including network-generation overhead."
    )
else:
    efficiency_statement = (
        "The optimized network did not reduce total runtime relative to the non-optimized network "
        "after including network-generation overhead. Computational efficiency should therefore "
        "not be claimed as a primary outcome for this dataset."
    )

print("\nComputational interpretation")
print("Pair reduction relative to non-optimized network: {:.2f}%".format(opt_pair_reduction))
print("Runtime reduction with overhead relative to non-optimized network: {:.2f}%".format(opt_runtime_reduction))
print(efficiency_statement)


with open(os.path.join(OUTPUT_DIR, "computational_interpretation.txt"), "w") as f:
    f.write("Pair reduction relative to non-optimized network: {:.2f}%\n".format(opt_pair_reduction))
    f.write("Runtime reduction with overhead relative to non-optimized network: {:.2f}%\n".format(opt_runtime_reduction))
    f.write(efficiency_statement + "\n")


# ------------------------------------------------------------
# 12. Velocity map comparison
# ------------------------------------------------------------

velocity_nonopt = final_result_nonopt["velocity"]
velocity_opt = final_result_opt["velocity"]

zmin, zmax = np.nanquantile(
    xr.concat([velocity_nonopt, velocity_opt], dim="network"),
    [0.01, 0.99]
)

zlim = max(abs(zmin), abs(zmax))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300)

velocity_nonopt.plot.imshow(
    cmap="turbo",
    vmin=-zlim,
    vmax=zlim,
    ax=axes[0]
)
axes[0].set_title("Non-Optimized SBAS Velocity")
axes[0].set_aspect("auto")

velocity_opt.plot.imshow(
    cmap="turbo",
    vmin=-zlim,
    vmax=zlim,
    ax=axes[1]
)
axes[1].set_title("Optimized SBAS Velocity")
axes[1].set_aspect("auto")

plt.suptitle("Velocity Comparison, mm/year")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "velocity_comparison_nonoptimized_vs_optimized.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 13. POI time-series comparison
# ------------------------------------------------------------

def extract_poi_timeseries(disp, poi_gdf):
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(poi_gdf).geometry][0]
    return disp.sel(x=x, y=y, method="nearest")


ts_nonopt = extract_poi_timeseries(final_result_nonopt["disp"], POI)
ts_opt = extract_poi_timeseries(final_result_opt["disp"], POI)

plt.figure(figsize=(10, 4), dpi=300)
plt.plot(ts_nonopt.date, ts_nonopt, label="Non-optimized network", lw=2)
plt.plot(ts_opt.date, ts_opt, label="Optimized network", lw=2)
plt.xlabel("Date")
plt.ylabel("LOS displacement [mm]")
plt.title("POI Displacement Time-Series Comparison")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "poi_timeseries_nonoptimized_vs_optimized.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 14. Velocity difference map
# ------------------------------------------------------------

velocity_diff = velocity_opt - velocity_nonopt

plt.figure(figsize=(6, 4), dpi=300)
velocity_diff.plot.imshow(cmap="turbo")
plt.title("Velocity Difference: Optimized minus Non-Optimized")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "velocity_difference_optimized_minus_nonoptimized.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 15. Export displacement and velocity outputs
# ------------------------------------------------------------

with dask.config.set(scheduler="synchronous"):
    final_result_nonopt["disp"].to_netcdf(
        os.path.join(OUTPUT_DIR, "disp_sbas_nonoptimized.nc")
    )
    final_result_opt["disp"].to_netcdf(
        os.path.join(OUTPUT_DIR, "disp_sbas_optimized.nc")
    )
    final_result_nonopt["velocity"].to_netcdf(
        os.path.join(OUTPUT_DIR, "velocity_sbas_nonoptimized.nc")
    )
    final_result_opt["velocity"].to_netcdf(
        os.path.join(OUTPUT_DIR, "velocity_sbas_optimized.nc")
    )


# ------------------------------------------------------------
# 16. Final saved outputs
# ------------------------------------------------------------

print("\nSaved outputs in:", OUTPUT_DIR)
print("network_optimization_sensitivity_limits.csv")
print("network_comparison_nonoptimized_vs_optimized.csv")
print("runtime_all_repeats_nonoptimized_vs_optimized.csv")
print("computational_efficiency_summary_nonoptimized_vs_optimized.csv")
print("computational_interpretation.txt")
print("disp_sbas_nonoptimized.nc")
print("disp_sbas_optimized.nc")
print("velocity_sbas_nonoptimized.nc")
print("velocity_sbas_optimized.nc")

In [ ]:
# ============================================================
# Optimized vs Non-Optimized SBAS Network Comparison
# Fair Computational Timing + Sensitivity Analysis
# Insert after:
#     sbas.compute_ps()
# ============================================================

import os
import gc
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import dask
import xarray as xr

# ------------------------------------------------------------
# 0. User-defined SBAS constraints
# ------------------------------------------------------------

MAX_DAYS_NONOPT = 60
MAX_BPERP_NONOPT = 150

MAX_DAYS_OPT = 60
MAX_BPERP_OPT = 150

LIMIT_VALUES = [1, 2, 3, 4, 5]
N_ITERATIONS = 3
OPTIMAL_LIMIT = 2

N_RUNTIME_REPEATS = 2

OUTPUT_DIR = "sbas_network_comparison_outputs_DESC"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 1. Utility functions
# ------------------------------------------------------------

def pairs_to_graph(pairs):
    G = nx.Graph()

    if hasattr(pairs, "columns"):
        cols = list(pairs.columns)

        if "ref" in cols and "rep" in cols:
            ref_col, rep_col = "ref", "rep"
        elif "reference" in cols and "repeat" in cols:
            ref_col, rep_col = "reference", "repeat"
        elif "date1" in cols and "date2" in cols:
            ref_col, rep_col = "date1", "date2"
        else:
            ref_col, rep_col = cols[0], cols[1]

        for _, row in pairs.iterrows():
            G.add_edge(str(row[ref_col]), str(row[rep_col]))
    else:
        for p in pairs:
            G.add_edge(str(p[0]), str(p[1]))

    return G


def network_statistics(G, pairs, name, limit_value=None, runtime=None):
    degrees = np.array([d for _, d in G.degree()])

    return {
        "network": name,
        "limit": limit_value,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "connected": nx.is_connected(G) if G.number_of_nodes() > 0 else False,
        "components": nx.number_connected_components(G) if G.number_of_nodes() > 0 else np.nan,
        "average_degree": np.mean(degrees) if len(degrees) > 0 else np.nan,
        "minimum_degree": np.min(degrees) if len(degrees) > 0 else np.nan,
        "maximum_degree": np.max(degrees) if len(degrees) > 0 else np.nan,
        "max_abs_baseline_m": np.abs(pairs["baseline"]).max() if "baseline" in pairs.columns else np.nan,
        "mean_abs_baseline_m": np.abs(pairs["baseline"]).mean() if "baseline" in pairs.columns else np.nan,
        "max_duration_days": pairs["duration"].max() if "duration" in pairs.columns else np.nan,
        "mean_duration_days": pairs["duration"].mean() if "duration" in pairs.columns else np.nan,
        "network_generation_runtime_s": runtime
    }


def remove_existing_stack(stack_name):
    possible_paths = [
        stack_name,
        f"{stack_name}.grd",
        f"{stack_name}.nc",
        os.path.join(OUTPUT_DIR, stack_name),
    ]

    for path in possible_paths:
        if os.path.isdir(path):
            shutil.rmtree(path, ignore_errors=True)
        elif os.path.isfile(path):
            os.remove(path)


def clean_memory():
    gc.collect()


# ------------------------------------------------------------
# 2. Generate non-optimized and optimized candidate networks
# ------------------------------------------------------------

t0 = time.perf_counter()
baseline_pairs_nonopt = sbas.sbas_pairs(
    days=MAX_DAYS_NONOPT,
    meters=MAX_BPERP_NONOPT
)
runtime_nonopt_pair_selection = time.perf_counter() - t0

t0 = time.perf_counter()
baseline_pairs_opt_raw = sbas.sbas_pairs(
    days=MAX_DAYS_OPT,
    meters=MAX_BPERP_OPT
)
runtime_opt_raw_pair_selection = time.perf_counter() - t0

print("Non-optimized candidate pairs:", len(baseline_pairs_nonopt))
print("Optimized raw candidate pairs:", len(baseline_pairs_opt_raw))


# ------------------------------------------------------------
# 3. Sensitivity analysis for redundancy-control limit
# ------------------------------------------------------------

sensitivity_rows = []
optimized_pairs_by_limit = {}
optimization_runtime_by_limit = {}

for limit_value in LIMIT_VALUES:
    print(f"\nRunning network sensitivity for limit = {limit_value}")

    t0 = time.perf_counter()

    pairs_limited = sbas.sbas_pairs_limit(
        baseline_pairs_opt_raw,
        limit=limit_value,
        iterations=N_ITERATIONS
    )

    runtime_limit = time.perf_counter() - t0

    G_limited = pairs_to_graph(pairs_limited)

    stats = network_statistics(
        G_limited,
        pairs_limited,
        name=f"Optimized_limit_{limit_value}",
        limit_value=limit_value,
        runtime=runtime_limit
    )

    sensitivity_rows.append(stats)
    optimized_pairs_by_limit[limit_value] = pairs_limited
    optimization_runtime_by_limit[limit_value] = runtime_limit

df_sensitivity = pd.DataFrame(sensitivity_rows)

df_sensitivity_out = df_sensitivity[
    [
        "limit",
        "nodes",
        "edges",
        "components",
        "connected",
        "average_degree",
        "minimum_degree",
        "maximum_degree",
        "max_abs_baseline_m",
        "mean_abs_baseline_m",
        "max_duration_days",
        "mean_duration_days",
        "network_generation_runtime_s"
    ]
]

df_sensitivity_out.to_csv(
    os.path.join(OUTPUT_DIR, "network_optimization_sensitivity_limits.csv"),
    index=False
)

print("\nNetwork optimization sensitivity table")
print(df_sensitivity_out)


# ------------------------------------------------------------
# 4. Select optimized network
# ------------------------------------------------------------

baseline_pairs_opt = optimized_pairs_by_limit[OPTIMAL_LIMIT]
runtime_opt_limit_selection = optimization_runtime_by_limit[OPTIMAL_LIMIT]

print("\nSelected optimized network")
print("Optimal limit:", OPTIMAL_LIMIT)
print("Optimized pairs:", len(baseline_pairs_opt))


# ------------------------------------------------------------
# 5. Network diagnostics
# ------------------------------------------------------------

G_nonopt = pairs_to_graph(baseline_pairs_nonopt)
G_opt = pairs_to_graph(baseline_pairs_opt)

stats_nonopt = network_statistics(
    G_nonopt,
    baseline_pairs_nonopt,
    "Non-optimized",
    limit_value=np.nan,
    runtime=runtime_nonopt_pair_selection
)

stats_opt = network_statistics(
    G_opt,
    baseline_pairs_opt,
    f"Optimized_limit_{OPTIMAL_LIMIT}",
    limit_value=OPTIMAL_LIMIT,
    runtime=runtime_opt_raw_pair_selection + runtime_opt_limit_selection
)

df_network_comparison = pd.DataFrame([stats_nonopt, stats_opt])

df_network_comparison_out = df_network_comparison[
    [
        "network",
        "limit",
        "nodes",
        "edges",
        "components",
        "connected",
        "average_degree",
        "minimum_degree",
        "maximum_degree",
        "max_abs_baseline_m",
        "mean_abs_baseline_m",
        "max_duration_days",
        "mean_duration_days",
        "network_generation_runtime_s"
    ]
]

df_network_comparison_out.to_csv(
    os.path.join(OUTPUT_DIR, "network_comparison_nonoptimized_vs_optimized.csv"),
    index=False
)

print("\nNetwork comparison table")
print(df_network_comparison_out)


# ------------------------------------------------------------
# 6. Plot sensitivity metrics
# ------------------------------------------------------------

plt.figure(figsize=(8, 4), dpi=300)
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["components"], marker="o")
plt.xlabel("Redundancy-control limit")
plt.ylabel("Number of connected components")
plt.title("Connected Components vs Redundancy-Control Limit")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sensitivity_components_vs_limit.png"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4), dpi=300)
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["average_degree"], marker="o", label="Average degree")
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["minimum_degree"], marker="s", label="Minimum degree")
plt.xlabel("Redundancy-control limit")
plt.ylabel("Node degree")
plt.title("Network Redundancy vs Redundancy-Control Limit")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sensitivity_degree_vs_limit.png"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4), dpi=300)
plt.plot(df_sensitivity_out["limit"], df_sensitivity_out["network_generation_runtime_s"], marker="o")
plt.xlabel("Redundancy-control limit")
plt.ylabel("Optimization runtime [s]")
plt.title("Network Optimization Runtime")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sensitivity_runtime_vs_limit.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 7. Plot baseline networks
# ------------------------------------------------------------

with mpl_settings({"figure.dpi": 300}):
    sbas.plot_baseline(baseline_pairs_nonopt)
    plt.title(
        f"Non-Optimized SBAS Network\n"
        f"Temporal <= {MAX_DAYS_NONOPT} days, |Bperp| <= {MAX_BPERP_NONOPT} m"
    )
    plt.savefig(os.path.join(OUTPUT_DIR, "baseline_network_nonoptimized.png"), dpi=300, bbox_inches="tight")
    plt.show()

with mpl_settings({"figure.dpi": 300}):
    sbas.plot_baseline(baseline_pairs_opt)
    plt.title(
        f"Optimized SBAS Network\n"
        f"Limit = {OPTIMAL_LIMIT}, Iterations = {N_ITERATIONS}, "
        f"Temporal <= {MAX_DAYS_OPT} days, |Bperp| <= {MAX_BPERP_OPT} m"
    )
    plt.savefig(os.path.join(OUTPUT_DIR, "baseline_network_optimized.png"), dpi=300, bbox_inches="tight")
    plt.show()


# ------------------------------------------------------------
# 8. Full SBAS processing function with timing
# ------------------------------------------------------------

def run_sbas_network(pairs, label, run_id):
    print(f"\nRunning SBAS network: {label}, run {run_id}")

    timings = {}

    intf_name = f"intf_mlook_{label}_run{run_id}"
    unwrap_name = f"unwrap_{label}_run{run_id}"
    trend_name = f"trend_{label}_run{run_id}"
    disp_name = f"disp_{label}_run{run_id}"
    velocity_name = f"velocity_{label}_run{run_id}"

    for name in [intf_name, unwrap_name, trend_name, disp_name, velocity_name]:
        remove_existing_stack(name)

    clean_memory()

    total_start = time.perf_counter()

    # Interferogram generation
    t0 = time.perf_counter()

    sbas.compute_interferogram_multilook(
        pairs,
        intf_name,
        wavelength=30,
        weight=sbas.psfunction()
    )

    ds = sbas.open_stack(intf_name)
    intf = ds.phase
    corr = ds.correlation

    timings["interferogram_generation_s"] = time.perf_counter() - t0

    # Phase unwrapping
    t0 = time.perf_counter()

    unwrap = sbas.unwrap_snaphu(intf, corr)
    unwrap = sbas.sync_cube(unwrap, unwrap_name)

    timings["phase_unwrapping_s"] = time.perf_counter() - t0

    # Trend correction
    t0 = time.perf_counter()

    decimator = sbas.decimator(resolution=15, grid=(1, 1))
    topo = decimator(sbas.get_topo())
    inc = decimator(sbas.incidence_angle())
    yy, xx = xr.broadcast(topo.y, topo.x)

    trend = sbas.regression(
        unwrap.phase,
        [
            topo, topo * yy, topo * xx, topo * yy * xx,
            topo**2, topo**2 * yy, topo**2 * xx, topo**2 * yy * xx,
            topo**3, topo**3 * yy, topo**3 * xx, topo**3 * yy * xx,
            inc, inc * yy, inc * xx, inc * yy * xx,
            yy, xx,
            yy**2, xx**2, yy * xx,
            yy**3, xx**3, yy**2 * xx, xx**2 * yy
        ],
        corr
    )

    trend = sbas.sync_cube(trend, trend_name)

    timings["trend_correction_s"] = time.perf_counter() - t0

    # Least-squares inversion
    t0 = time.perf_counter()

    disp = sbas.los_displacement_mm(
        sbas.lstsq(unwrap.phase - trend, corr)
    )

    disp = sbas.sync_cube(disp, disp_name)

    timings["time_series_inversion_s"] = time.perf_counter() - t0

    # Velocity estimation
    t0 = time.perf_counter()

    velocity = sbas.velocity(disp)
    velocity = sbas.sync_cube(velocity, velocity_name)

    timings["velocity_estimation_s"] = time.perf_counter() - t0

    timings["total_processing_s"] = time.perf_counter() - total_start

    return {
        "label": label,
        "run_id": run_id,
        "pairs": pairs,
        "intf": intf,
        "corr": corr,
        "unwrap": unwrap,
        "trend": trend,
        "disp": disp,
        "velocity": velocity,
        "timings": timings
    }


# ------------------------------------------------------------
# ------------------------------------------------------------
# 9. Fair runtime experiment
# Optimized network is always run before non-optimized network
# ------------------------------------------------------------

runtime_results = []
final_result_nonopt = None
final_result_opt = None

for run_id in range(1, N_RUNTIME_REPEATS + 1):

    run_order = [
        ("opt", baseline_pairs_opt, f"Optimized_limit_{OPTIMAL_LIMIT}"),
        ("nonopt", baseline_pairs_nonopt, "Non-optimized")
    ]

    for order_position, (short_label, pairs, network_name) in enumerate(run_order, start=1):

        result = run_sbas_network(pairs, short_label, run_id)

        row = {
            "network": network_name,
            "run_id": run_id,
            "run_order_position": order_position,
            "pairs": len(pairs)
        }

        row.update(result["timings"])
        runtime_results.append(row)

        if short_label == "nonopt":
            final_result_nonopt = result
        else:
            final_result_opt = result

df_runtime_all = pd.DataFrame(runtime_results)

df_runtime_all.to_csv(
    os.path.join(OUTPUT_DIR, "runtime_all_repeats_nonoptimized_vs_optimized.csv"),
    index=False
)

print("\nAll runtime repeats")
print(df_runtime_all)


# ------------------------------------------------------------
# 10. Runtime summary
# ------------------------------------------------------------

timing_columns = [
    "interferogram_generation_s",
    "phase_unwrapping_s",
    "trend_correction_s",
    "time_series_inversion_s",
    "velocity_estimation_s",
    "total_processing_s"
]

df_runtime_summary = df_runtime_all.groupby("network").agg(
    pairs=("pairs", "mean"),
    **{f"{col}_mean": (col, "mean") for col in timing_columns},
    **{f"{col}_std": (col, "std") for col in timing_columns}
).reset_index()

nonopt_total_mean = df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "total_processing_s_mean"
].iloc[0]

nonopt_pairs_mean = df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "pairs"
].iloc[0]

df_runtime_summary["runtime_reduction_relative_to_nonopt_percent"] = (
    (nonopt_total_mean - df_runtime_summary["total_processing_s_mean"])
    / nonopt_total_mean * 100.0
)

df_runtime_summary["pair_reduction_relative_to_nonopt_percent"] = (
    (nonopt_pairs_mean - df_runtime_summary["pairs"])
    / nonopt_pairs_mean * 100.0
)

# Add pair-selection and optimization overhead
df_runtime_summary["network_generation_overhead_s"] = np.nan

df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "network_generation_overhead_s"
] = runtime_nonopt_pair_selection

df_runtime_summary.loc[
    df_runtime_summary["network"] == f"Optimized_limit_{OPTIMAL_LIMIT}",
    "network_generation_overhead_s"
] = runtime_opt_raw_pair_selection + runtime_opt_limit_selection

df_runtime_summary["total_processing_with_network_generation_s"] = (
    df_runtime_summary["total_processing_s_mean"]
    + df_runtime_summary["network_generation_overhead_s"]
)

nonopt_total_with_overhead = df_runtime_summary.loc[
    df_runtime_summary["network"] == "Non-optimized",
    "total_processing_with_network_generation_s"
].iloc[0]

df_runtime_summary["runtime_reduction_with_overhead_relative_to_nonopt_percent"] = (
    (nonopt_total_with_overhead - df_runtime_summary["total_processing_with_network_generation_s"])
    / nonopt_total_with_overhead * 100.0
)

df_runtime_summary.to_csv(
    os.path.join(OUTPUT_DIR, "computational_efficiency_summary_nonoptimized_vs_optimized.csv"),
    index=False
)

print("\nRuntime summary")
print(df_runtime_summary)


# ------------------------------------------------------------
# 11. Interpretation flag
# ------------------------------------------------------------

opt_name = f"Optimized_limit_{OPTIMAL_LIMIT}"

opt_runtime_reduction = df_runtime_summary.loc[
    df_runtime_summary["network"] == opt_name,
    "runtime_reduction_with_overhead_relative_to_nonopt_percent"
].iloc[0]

opt_pair_reduction = df_runtime_summary.loc[
    df_runtime_summary["network"] == opt_name,
    "pair_reduction_relative_to_nonopt_percent"
].iloc[0]

if opt_runtime_reduction > 0:
    efficiency_statement = (
        "The optimized network reduced total runtime relative to the non-optimized network "
        "after including network-generation overhead."
    )
else:
    efficiency_statement = (
        "The optimized network did not reduce total runtime relative to the non-optimized network "
        "after including network-generation overhead. Computational efficiency should therefore "
        "not be claimed as a primary outcome for this dataset."
    )

print("\nComputational interpretation")
print("Pair reduction relative to non-optimized network: {:.2f}%".format(opt_pair_reduction))
print("Runtime reduction with overhead relative to non-optimized network: {:.2f}%".format(opt_runtime_reduction))
print(efficiency_statement)


with open(os.path.join(OUTPUT_DIR, "computational_interpretation.txt"), "w") as f:
    f.write("Pair reduction relative to non-optimized network: {:.2f}%\n".format(opt_pair_reduction))
    f.write("Runtime reduction with overhead relative to non-optimized network: {:.2f}%\n".format(opt_runtime_reduction))
    f.write(efficiency_statement + "\n")


# ------------------------------------------------------------
# 12. Velocity map comparison
# ------------------------------------------------------------

velocity_nonopt = final_result_nonopt["velocity"]
velocity_opt = final_result_opt["velocity"]

zmin, zmax = np.nanquantile(
    xr.concat([velocity_nonopt, velocity_opt], dim="network"),
    [0.01, 0.99]
)

zlim = max(abs(zmin), abs(zmax))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300)

velocity_nonopt.plot.imshow(
    cmap="turbo",
    vmin=-zlim,
    vmax=zlim,
    ax=axes[0]
)
axes[0].set_title("Non-Optimized SBAS Velocity")
axes[0].set_aspect("auto")

velocity_opt.plot.imshow(
    cmap="turbo",
    vmin=-zlim,
    vmax=zlim,
    ax=axes[1]
)
axes[1].set_title("Optimized SBAS Velocity")
axes[1].set_aspect("auto")

plt.suptitle("Velocity Comparison, mm/year")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "velocity_comparison_nonoptimized_vs_optimized.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 13. POI time-series comparison
# ------------------------------------------------------------

def extract_poi_timeseries(disp, poi_gdf):
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(poi_gdf).geometry][0]
    return disp.sel(x=x, y=y, method="nearest")


ts_nonopt = extract_poi_timeseries(final_result_nonopt["disp"], POI)
ts_opt = extract_poi_timeseries(final_result_opt["disp"], POI)

plt.figure(figsize=(10, 4), dpi=300)
plt.plot(ts_nonopt.date, ts_nonopt, label="Non-optimized network", lw=2)
plt.plot(ts_opt.date, ts_opt, label="Optimized network", lw=2)
plt.xlabel("Date")
plt.ylabel("LOS displacement [mm]")
plt.title("POI Displacement Time-Series Comparison")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "poi_timeseries_nonoptimized_vs_optimized.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 14. Velocity difference map
# ------------------------------------------------------------

velocity_diff = velocity_opt - velocity_nonopt

plt.figure(figsize=(6, 4), dpi=300)
velocity_diff.plot.imshow(cmap="turbo")
plt.title("Velocity Difference: Optimized minus Non-Optimized")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "velocity_difference_optimized_minus_nonoptimized.png"), dpi=300, bbox_inches="tight")
plt.show()


# ------------------------------------------------------------
# 15. Export displacement and velocity outputs
# ------------------------------------------------------------

with dask.config.set(scheduler="synchronous"):
    final_result_nonopt["disp"].to_netcdf(
        os.path.join(OUTPUT_DIR, "disp_sbas_nonoptimized.nc")
    )
    final_result_opt["disp"].to_netcdf(
        os.path.join(OUTPUT_DIR, "disp_sbas_optimized.nc")
    )
    final_result_nonopt["velocity"].to_netcdf(
        os.path.join(OUTPUT_DIR, "velocity_sbas_nonoptimized.nc")
    )
    final_result_opt["velocity"].to_netcdf(
        os.path.join(OUTPUT_DIR, "velocity_sbas_optimized.nc")
    )


# ------------------------------------------------------------
# 16. Final saved outputs
# ------------------------------------------------------------

print("\nSaved outputs in:", OUTPUT_DIR)
print("network_optimization_sensitivity_limits.csv")
print("network_comparison_nonoptimized_vs_optimized.csv")
print("runtime_all_repeats_nonoptimized_vs_optimized.csv")
print("computational_efficiency_summary_nonoptimized_vs_optimized.csv")
print("computational_interpretation.txt")
print("disp_sbas_nonoptimized.nc")
print("disp_sbas_optimized.nc")
print("velocity_sbas_nonoptimized.nc")
print("velocity_sbas_optimized.nc")

In [ ]:
## SBAS Analysis

In [ ]:
### Multi-looked Resolution for SBAS

In [ ]:
sbas.compute_interferogram_multilook(baseline_pairs, 'intf_mlook', wavelength=30, weight=sbas.psfunction())

In [ ]:
ds_sbas = sbas.open_stack('intf_mlook')
intf_sbas = ds_sbas.phase
corr_sbas = ds_sbas.correlation
corr_sbas

In [ ]:
sbas.plot_interferograms(intf_sbas[:8], caption='SBAS Phase, [rad]')

In [ ]:
sbas.plot_correlations(corr_sbas[:8], caption='SBAS Correlation')

In [ ]:
unwrap_sbas = sbas.unwrap_snaphu(intf_sbas, corr_sbas)
unwrap_sbas

In [ ]:
# optionally, materialize to disk and open
unwrap_sbas = sbas.sync_cube(unwrap_sbas, 'unwrap_sbas')

In [ ]:
decimator = sbas.decimator(resolution=15, grid=(1,1))
topo = decimator(sbas.get_topo())
inc = decimator(sbas.incidence_angle())
yy, xx = xr.broadcast(topo.y, topo.x)
trend_sbas = sbas.regression(unwrap_sbas.phase,
        [topo,    topo*yy,    topo*xx,    topo*yy*xx,
         topo**2, topo**2*yy, topo**2*xx, topo**2*yy*xx,
         topo**3, topo**3*yy, topo**3*xx, topo**3*yy*xx,
         inc,     inc**yy,    inc*xx,     inc*yy*xx,
         yy, xx,
         yy**2, xx**2, yy*xx,
         yy**3, xx**3, yy**2*xx, xx**2*yy], corr_sbas)

In [ ]:
# optionally, materialize to disk and open
trend_sbas = sbas.sync_cube(trend_sbas, 'trend_sbas')

In [ ]:
sbas.plot_phases(trend_sbas[:8], caption='SBAS Trend Phase, [rad]', quantile=[0.01, 0.99])

In [ ]:
sbas.plot_phases((unwrap_sbas.phase - trend_sbas)[:8], caption='SBAS Phase - Trend, [rad]', vmin=-np.pi, vmax=np.pi)

In [ ]:
# calculate phase displacement in radians and convert to LOS displacement in millimeter
disp_sbas = sbas.los_displacement_mm(sbas.lstsq(unwrap_sbas.phase - trend_sbas, corr_sbas))

In [ ]:
# optionally, materialize to disk and open
disp_sbas = sbas.sync_cube(disp_sbas, 'disp_sbas')

In [ ]:
velocity_sbas = sbas.velocity(disp_sbas)
velocity_sbas

In [ ]:
# optionally, materialize to disk and open
velocity_sbas = sbas.sync_cube(velocity_sbas, 'velocity_sbas')

In [ ]:
fig = plt.figure(figsize=(12,4), dpi=300)

zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

ax = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
sbas.geocode(AOI.buffer(-BUFFER).boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
sbas.geocode(POI0).plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

ax = fig.add_subplot(1, 2, 2)
sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry.buffer(-BUFFER))\
    .plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
AOI.buffer(-BUFFER).boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
POI0.plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

plt.suptitle('SBAS LOS Velocity, 2021', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4), dpi=300)

x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI).geometry][0]
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)
plt.plot(disp_pixel.date, disp_pixel, c='r', lw=2, label='Displacement POI')
plt.plot(stl_pixel.date, stl_pixel.trend, c='r', ls='--', lw=2, label='Trend POI')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='r', lw=1, label='Seasonal POI')

x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI0).geometry][0]
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)
plt.plot(disp_pixel.date, disp_pixel, c='b', lw=2, label='Displacement POI$\Theta$')
plt.plot(stl_pixel.date, stl_pixel.trend, c='b', ls='--', lw=2, label='Trend POI$\Theta$')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='b', lw=1, label='Seasonal POI$\Theta$')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=14)
plt.title('SBAS LOS Displacement STL Decompose, 2021', fontsize=18)
plt.ylabel('Displacement, mm', fontsize=16)
plt.show()

In [ ]:
import json
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Define all POIs in one block
# -----------------------------

# Assign distinct colors (auto-cycled)
colors = plt.cm.tab20(np.linspace(0, 1, len(points)))

# -----------------------------
# Create the plot
# -----------------------------
plt.figure(figsize=(14, 6), dpi=300)

for (name, coord), color in zip(points.items(), colors):

    # Build GeoDataFrame dynamically
    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }
    gdf = gpd.GeoDataFrame.from_features([geojson])

    # Get nearest pixel
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    # Extract displacement & STL
    disp_px = disp_sbas.sel(x=x, y=y, method="nearest")
    stl_px = sbas.stl(disp_sbas.sel(x=[x], y=[y], method="nearest")).isel(x=0, y=0)

    # Plot everything with same color for each POI
    plt.plot(disp_px.date, disp_px, lw=1.8, color=color, label=f"{name} Displacement")
    #plt.plot(stl_px.date, stl_px.trend, lw=1.2, ls="--", color=color, alpha=0.9)
    #plt.plot(stl_px.date, stl_px.seasonal, lw=0.8, ls=":", color=color, alpha=0.7)

# -----------------------------
# Final plot formatting
# -----------------------------
plt.title("SBAS Displacement Time Series for All POIs")
plt.xlabel("Timeline")
plt.ylabel("Displacement [mm]")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(ncol=2, fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------
# Select only specific POIs
# ---------------------------------
selected_pois = ["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12","A13","A14","A15","A16","A17","A18","A19","A20","A21","A22","A23","A24","A25","A26","A27","A28","A29","A30","A31","A32","A33","A34","A35","A36","A37","A38","A39","A40","A41","A42","A43","A44","A45","A46"]   # <-- put desired names here

# Filter dictionary
filtered_points = {k: v for k, v in points.items() if k in selected_pois}

colors = plt.cm.tab20(np.linspace(0, 1, len(filtered_points)))

plt.figure(figsize=(14, 6), dpi=300)

for (name, coord), color in zip(filtered_points.items(), colors):

    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }

    gdf = gpd.GeoDataFrame.from_features([geojson])
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    disp_px = disp_sbas.sel(x=x, y=y, method="nearest")

    plt.plot(disp_px.date, disp_px,
             lw=1.8,
             color=color,
             label=f"{name} Displacement")

plt.title("SBAS Displacement Time Series for Selected POIs")
plt.xlabel("Timeline")
plt.ylabel("Displacement [mm]")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(ncol=2, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
import dask

with dask.config.set(scheduler="synchronous"):
    disp_sbas.to_netcdf(
        "disp_sbas_CHNA_2016_2019_Desc_B60.nc"
    )



In [ ]:
import geopandas as gpd
import pandas as pd
import json
import numpy as np

# -------------------------------
# Geometry grids from PyGMTSAR
# -------------------------------

# incidence angle (radians)
inc = sbas.incidence_angle()

# satellite look vector
lv = sbas.get_satellite_look_vector()
look_E = lv["look_E"]
look_N = lv["look_N"]
look_U = lv["look_U"]

records = []

for name, coord in points.items():

    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }
    gdf = gpd.GeoDataFrame.from_features([geojson])

    # lon/lat → radar x,y
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    # sample incidence angle
    inc_rad = inc.sel(x=x, y=y, method="nearest").values.item()
    inc_deg = np.rad2deg(inc_rad)

    # sample look vector components
    le = look_E.sel(x=x, y=y, method="nearest").values.item()
    ln = look_N.sel(x=x, y=y, method="nearest").values.item()
    lu = look_U.sel(x=x, y=y, method="nearest").values.item()

    records.append({
        "point": name,
        "lon": coord[0],
        "lat": coord[1],
        "x": x,
        "y": y,

        # incidence
        "incidence_rad": inc_rad,
        "incidence_deg": inc_deg,

        # look vector components (ENU)
        "look_E": le,
        "look_N": ln,
        "look_U": lu
    })

df_geom_asc = pd.DataFrame(records)

# save for later restore-only use
df_geom_asc.to_csv(
    "Apoi_geometryALLPoints_angles_CHNA_2016_2019_ASC_B60_Apr12.csv",
    index=False
)




In [ ]:
fig = plt.figure(figsize=(12,4), dpi=300)

zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

ax = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
sbas.geocode(AOI.buffer(-BUFFER).boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
sbas.geocode(POI0).plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

ax = fig.add_subplot(1, 2, 2)
sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry.buffer(-BUFFER))\
    .plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
AOI.buffer(-BUFFER).boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
POI0.plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

plt.suptitle('SBAS LOS Velocity, 2021', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
# optionally, materialize to disk and open
disp_sbas = sbas.sync_cube(disp_sbas, 'disp_sbas')

In [ ]:
sbas.plot_displacements(disp_sbas[::3], caption='SBAS Cumulative LOS Displacement, [mm]', quantile=[0.01, 0.99])

In [ ]:
fig = plt.figure(figsize=(12,4), dpi=300)

zmin, zmax = np.nanquantile(velocity_sbas, [0.01, 0.99])
zminmax = max(abs(zmin), zmax)

ax = fig.add_subplot(1, 2, 1)
velocity_sbas.plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
sbas.geocode(AOI.buffer(-BUFFER).boundary).plot(ax=ax)
sbas.geocode(POI).plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
sbas.geocode(POI0).plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.set_aspect('auto')
ax.set_title('Velocity, mm/year', fontsize=16)

ax = fig.add_subplot(1, 2, 2)
sbas.as_geo(sbas.ra2ll(velocity_sbas)).rio.clip(AOI.geometry.buffer(-BUFFER))\
    .plot.imshow(cmap='turbo', vmin=-zminmax, vmax=zminmax, ax=ax)
AOI.buffer(-BUFFER).boundary.plot(ax=ax)
POI.plot(ax=ax, marker='x', c='r', markersize=100, label='POI')
POI0.plot(ax=ax, marker='x', c='b', markersize=100, label='POI$\Theta$')
ax.legend(loc='upper left', fontsize=14)
ax.set_title('Velocity, mm/year', fontsize=16)

plt.suptitle('SBAS LOS Velocity, 2021', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4), dpi=300)

x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI).geometry][0]
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)
plt.plot(disp_pixel.date, disp_pixel, c='r', lw=2, label='Displacement POI')
plt.plot(stl_pixel.date, stl_pixel.trend, c='r', ls='--', lw=2, label='Trend POI')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='r', lw=1, label='Seasonal POI')

x, y = [(geom.x, geom.y) for geom in sbas.geocode(POI0).geometry][0]
disp_pixel = disp_sbas.sel(y=y, x=x, method='nearest')
stl_pixel = sbas.stl(disp_sbas.sel(y=[y], x=[x], method='nearest')).isel(x=0, y=0)
plt.plot(disp_pixel.date, disp_pixel, c='b', lw=2, label='Displacement POI$\Theta$')
plt.plot(stl_pixel.date, stl_pixel.trend, c='b', ls='--', lw=2, label='Trend POI$\Theta$')
plt.plot(stl_pixel.date, stl_pixel.seasonal, c='b', lw=1, label='Seasonal POI$\Theta$')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=14)
plt.title('SBAS LOS Displacement STL Decompose, 2021', fontsize=18)
plt.ylabel('Displacement, mm', fontsize=16)
plt.show()

In [ ]:
import json
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Define all POIs in one block
# -----------------------------

# Assign distinct colors (auto-cycled)
colors = plt.cm.tab20(np.linspace(0, 1, len(points)))

# -----------------------------
# Create the plot
# -----------------------------
plt.figure(figsize=(14, 6), dpi=300)

for (name, coord), color in zip(points.items(), colors):

    # Build GeoDataFrame dynamically
    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }
    gdf = gpd.GeoDataFrame.from_features([geojson])

    # Get nearest pixel
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    # Extract displacement & STL
    disp_px = disp_sbas.sel(x=x, y=y, method="nearest")
    stl_px = sbas.stl(disp_sbas.sel(x=[x], y=[y], method="nearest")).isel(x=0, y=0)

    # Plot everything with same color for each POI
    plt.plot(disp_px.date, disp_px, lw=1.8, color=color, label=f"{name} Displacement")
    #plt.plot(stl_px.date, stl_px.trend, lw=1.2, ls="--", color=color, alpha=0.9)
    #plt.plot(stl_px.date, stl_px.seasonal, lw=0.8, ls=":", color=color, alpha=0.7)

# -----------------------------
# Final plot formatting
# -----------------------------
plt.title("SBAS Displacement Time Series for All POIs")
plt.xlabel("Timeline")
plt.ylabel("Displacement [mm]")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(ncol=2, fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------
# Select only specific POIs
# ---------------------------------
selected_pois = ["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12","A13","A14","A15","A16","A17","A18","A19","A20","A21","A22","A23","A24","A25","A26","A27","A28","A29","A30","A31","A32","A33","A34","A35","A36","A37","A38","A39","A40","A41","A42","A43","A44","A45","A46"]   # <-- put desired names here

# Filter dictionary
filtered_points = {k: v for k, v in points.items() if k in selected_pois}

colors = plt.cm.tab20(np.linspace(0, 1, len(filtered_points)))

plt.figure(figsize=(14, 6), dpi=300)

for (name, coord), color in zip(filtered_points.items(), colors):

    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }

    gdf = gpd.GeoDataFrame.from_features([geojson])
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    disp_px = disp_sbas.sel(x=x, y=y, method="nearest")

    plt.plot(disp_px.date, disp_px,
             lw=1.8,
             color=color,
             label=f"{name} Displacement")

plt.title("SBAS Displacement Time Series for Selected POIs")
plt.xlabel("Timeline")
plt.ylabel("Displacement [mm]")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(ncol=2, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
import dask

with dask.config.set(scheduler="synchronous"):
    disp_sbas.to_netcdf(
        "disp_sbas_CHNA_2016_2019_Desc_B60.nc"
    )


In [ ]:
import geopandas as gpd
import pandas as pd
import json
import numpy as np

# -------------------------------
# Geometry grids from PyGMTSAR
# -------------------------------

# incidence angle (radians)
inc = sbas.incidence_angle()

# satellite look vector
lv = sbas.get_satellite_look_vector()
look_E = lv["look_E"]
look_N = lv["look_N"]
look_U = lv["look_U"]

records = []

for name, coord in points.items():

    geojson = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": coord},
        "properties": {}
    }
    gdf = gpd.GeoDataFrame.from_features([geojson])

    # lon/lat → radar x,y
    x, y = [(geom.x, geom.y) for geom in sbas.geocode(gdf).geometry][0]

    # sample incidence angle
    inc_rad = inc.sel(x=x, y=y, method="nearest").values.item()
    inc_deg = np.rad2deg(inc_rad)

    # sample look vector components
    le = look_E.sel(x=x, y=y, method="nearest").values.item()
    ln = look_N.sel(x=x, y=y, method="nearest").values.item()
    lu = look_U.sel(x=x, y=y, method="nearest").values.item()

    records.append({
        "point": name,
        "lon": coord[0],
        "lat": coord[1],
        "x": x,
        "y": y,

        # incidence
        "incidence_rad": inc_rad,
        "incidence_deg": inc_deg,

        # look vector components (ENU)
        "look_E": le,
        "look_N": ln,
        "look_U": lu
    })

df_geom_asc = pd.DataFrame(records)

# save for later restore-only use
df_geom_asc.to_csv(
    "Apoi_geometryALLPoints_angles_CHNA_2016_2019_ASC_B60_Apr12.csv",
    index=False
)


